# Training: same-location 2D task (`BioLeakyRNNTopo` + `CuedTargetSameLoc2D`)

Cue and target at the SAME random 2D location (uniform over the disk). Outputs: binary
go/no-go (`W_out`) + spatial (x,y) readout `decode_xy` (aux MSE, weight `LAMBDA_SPATIAL`).
**No COM loss.** Stages 0-1 only (stage 2 / distractors deferred).
Checkpoints: `../checkpoints/stage{0,1}_topo_sameloc.pt`.

In [ ]:
import sys, os, pathlib
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
import torch, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter
from src.model_topo import BioLeakyRNNTopo
from src.env import CuedTargetSameLoc2D
from src.training import TrainConfig, train_supervised
from src.analysis import collect_trials_spatial, filter_trials
device = 'cpu'
Path('checkpoints').mkdir(exist_ok=True)
LAMBDA_SPATIAL = 0.5   # weight on the spatial (x,y) readout MSE (tunable)
print('device:', device, '| LAMBDA_SPATIAL =', LAMBDA_SPATIAL)

In [ ]:
def make_model():
    return BioLeakyRNNTopo(
        input_size=7, hidden_size=180, output_size=2,   # binary go/no-go (W_out); W_spatial=decode_xy
        dt=20.0, tau=100.0, activation='softplus', sigma_rec=0.10, rec_init='diag',
        use_ei=True, exc_ratio=0.80, use_dale=True, mask_seed=42, sheet_side=12,
        tau_ee=0.25, tau_ie=0.32, tau_ei=0.64, tau_ii=0.64, rf_sigma=0.3,
        tau_e_range=(50, 150), tau_i_range=(10, 50), use_adaptation=False,
    ).to(device)

## Stage 0 — detect target, no cue, no distractors

In [ ]:
def make_env_stage0():
    return CuedTargetSameLoc2D(dt=20, cue_strength=0.0, p_distractor_trial=0.0, distractor_strength=0.0)


model = make_model()
cfg = TrainConfig(
    batch_size=64, lr=1e-3,
    max_updates=1000, print_every=100, device=device,
    com_weight=0.0, sparsity_weight=0.0, fa_penalty_weight=0.0,   # NO COM/sparsity/FA
    aux_weight=LAMBDA_SPATIAL,   # spatial (x,y) readout MSE
    xy_mask_from='target',       # aux active during + after target
    stop_on_no_miss=0,           # no early stopping
)
history_stage0_topo_sameloc = train_supervised(model, make_env_stage0, cfg)
torch.save({'state_dict': model.state_dict()}, 'checkpoints/stage0_topo_sameloc.pt')
print('Saved: checkpoints/stage0_topo_sameloc.pt')

## Stage 1 — add cue, no distractors (with early stopping)

In [ ]:
def make_env_stage1():
    return CuedTargetSameLoc2D(dt=20, cue_strength=1.0, p_distractor_trial=0.0, distractor_strength=0.0)


model = make_model()
model.load_state_dict(
    torch.load('checkpoints/stage0_topo_sameloc.pt', weights_only=True)['state_dict'], strict=False)
cfg = TrainConfig(
    batch_size=64, lr=1e-3,
    max_updates=5000, print_every=50, device=device,
    com_weight=0.0, sparsity_weight=0.0, fa_penalty_weight=0.0,   # NO COM/sparsity/FA
    aux_weight=LAMBDA_SPATIAL,   # spatial (x,y) readout MSE
    xy_mask_from='target',       # aux active during + after target
    # early stopping ON (default stop_on_no_miss=3)
)
history_stage1_topo_sameloc = train_supervised(model, make_env_stage1, cfg)
torch.save({'state_dict': model.state_dict()}, 'checkpoints/stage1_topo_sameloc.pt')
print('Saved: checkpoints/stage1_topo_sameloc.pt')

## Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for hist, name in [(history_stage0_topo_sameloc, 'stage0'), (history_stage1_topo_sameloc, 'stage1')]:
    axes[0].plot(hist['ce'], label=name+' CE', alpha=0.7)
    axes[0].plot(hist['aux'], label=name+' aux', alpha=0.7, ls='--')
    axes[1].plot([v*100 for v in hist['p_correct']], label=name+' hit', alpha=0.8)
axes[0].set_xlabel('print step'); axes[0].set_ylabel('loss'); axes[0].set_title('CE (go/no-go) + aux (x,y) MSE'); axes[0].legend(fontsize=7)
axes[1].set_xlabel('print step'); axes[1].set_ylabel('% correct'); axes[1].set_title('Go/no-go hit rate'); axes[1].legend(fontsize=7)
plt.tight_layout(); plt.show()

## Quick eval on stage 1 (go/no-go head)

In [ ]:
trials = collect_trials_spatial(model, make_env_stage1, n_trials=500, device=device, head='go_nogo')
out_counts = Counter(tr['train_outcome'] for tr in trials)
total = len(trials); print('Total trials:', total)
for o, n in sorted(out_counts.items(), key=lambda x: -x[1]):
    print('  %s: %d (%.1f%%)' % (o, n, 100*n/total))
errs = [tr['mean_err'] for tr in trials if tr.get('mean_err') is not None]
if errs: print('Mean (x,y) decode error vs target: %.4f' % np.mean(errs))